# Mean Pooling

In [ ]:
!pip install -q sentence-transformers pandas tqdm openpyxl

import json
import os
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score

# ═══════════════════════════════════════════════════════════════════════════
# CẤU HÌNH ĐƯỜNG DẪN DATASET VIỆT NAM
# ═══════════════════════════════════════════════════════════════════════════
DATASET_ROOT = "/kaggle/input/datasets/thaiphonghuan/scoring/vietname-dataset-scoring - training"
CV_DIR = os.path.join(DATASET_ROOT, "cv")
JD_DIR = os.path.join(DATASET_ROOT, "jd")
TEST_MATCH_CSV = os.path.join(DATASET_ROOT, "dataset/test/match/test_match.csv")
TEST_MISMATCH_CSV = os.path.join(DATASET_ROOT, "dataset/test/mismatch/test_mismatch.csv")

CV_VIEWS = ['skills', 'education', 'year_experience', 'experience', 'responsibilities', 
            'projects', 'publications', 'achievement', 'certification', 'employment_history']
JD_VIEWS = ['skills', 'education', 'year_experience_requirement', 'experience_requirement', 'responsibilities']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Model đa ngôn ngữ hỗ trợ tiếng Việt cực tốt cho Bi-encoder
MODEL_NAME = 'BAAI/bge-base-en-v1.5'
sbert_model = SentenceTransformer(MODEL_NAME).to(device)

print(f"✅ Model loaded: {MODEL_NAME}")

In [ ]:
def process_view(view_data):
    """Xử lý missing_section thành special token [EMPTY_SECTION]; trả về (text, exists_flag)."""
    if view_data is None: return "[EMPTY_SECTION]", 0
    if isinstance(view_data, list):
        clean_list = [str(x) for x in view_data if str(x).lower() != "missing_section"]
        if not clean_list: return "[EMPTY_SECTION]", 0
        return ". ".join(clean_list), 1
    val = str(view_data).lower()
    if val in ("missing_section", "", "none"): return "[EMPTY_SECTION]", 0
    return str(view_data), 1

def run_mean_pooling_on_csv():
    all_scores = []
    y_true_reg = []
    y_true_bin = []
    
    configs = [(TEST_MATCH_CSV, 1), (TEST_MISMATCH_CSV, 0)]
    
    print(f"🚀 Đang chạy Multi-view Mean Pooling (Bi-Encoder) cho dữ liệu Việt Nam...")
    
    for csv_path, b_label in configs:
        if not os.path.exists(csv_path):
            print(f"⚠️ Không tìm thấy: {csv_path}")
            continue
            
        df = pd.read_csv(csv_path)
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"CSV {b_label}"):
            # 1. Load JSON với logic Fallback (Sửa lỗi khiến Sim = 0)
            with open(os.path.join(CV_DIR, row['cv']), 'r', encoding='utf-8-sig') as f:
                cv_obj = json.load(f)
                cv_j = cv_obj.get('resume_structured', cv_obj) # Lấy key hoặc lấy chính nó
            with open(os.path.join(JD_DIR, row['jd']), 'r', encoding='utf-8-sig') as f:
                jd_obj = json.load(f)
                jd_j = jd_obj.get('job_description_structured', jd_obj)

            # 2. Trích xuất văn bản và mặt nạ cho từng View
            cv_texts, cv_masks = zip(*[process_view(cv_j.get(v)) for v in CV_VIEWS])
            jd_texts, jd_masks = zip(*[process_view(jd_j.get(v)) for v in JD_VIEWS])
            
            with torch.no_grad():
                # 3. Mã hóa độc lập từng View thành các vector (Bi-Encoder)
                cv_emb = sbert_model.encode(list(cv_texts), convert_to_tensor=True) # [10, d]
                jd_emb = sbert_model.encode(list(jd_texts), convert_to_tensor=True) # [5, d]
                
                # 4. Thực hiện Masked Mean Pooling
                # Chuyển đổi mask sang tensor [N, 1]
                cv_mask_t = torch.tensor(cv_masks, device=device).unsqueeze(1)
                jd_mask_t = torch.tensor(jd_masks, device=device).unsqueeze(1)
                
                # Chỉ tính trung bình các view có dữ liệu (mask = 1)
                # z = sum(embeddings * mask) / count_active_views
                z_cv = (cv_emb * cv_mask_t).sum(dim=0) / (cv_mask_t.sum() + 1e-9)
                z_jd = (jd_emb * jd_mask_t).sum(dim=0) / (jd_mask_t.sum() + 1e-9)
                
                # 5. Tính Cosine Similarity giữa 2 vector đại diện duy nhất
                score = util.cos_sim(z_cv, z_jd).item()
            
            all_scores.append(score)
            y_true_reg.append(float(row['score']))
            y_true_bin.append(b_label)
            
    # Kiểm tra nhanh xem có bị lỗi Sim = 0 nữa không
    final_scores = np.array(all_scores)
    print(f"🔎 Sanity Check | Mean Sim: {final_scores.mean():.4f} | Max Sim: {final_scores.max():.4f}")
    
    return final_scores, np.array(y_true_reg), np.array(y_true_bin)

# Thực thi
y_pred_test, y_true_reg, y_true_bin = run_mean_pooling_on_csv()

In [ ]:
import openpyxl

# 1. Chuyển đổi điểm số sang nhãn nhị phân dựa trên ngưỡng 0.5
threshold = 0.5
y_pred_bin = (y_pred_test >= threshold).astype(int)

# 2. Tính toán trọn bộ Metrics
mae = mean_absolute_error(y_true_reg, y_pred_test)
mse = mean_squared_error(y_true_reg, y_pred_test)
acc = accuracy_score(y_true_bin, y_pred_bin)
pre = precision_score(y_true_bin, y_pred_bin, zero_division=0)
rec = recall_score(y_true_bin, y_pred_bin, zero_division=0)
f1  = f1_score(y_true_bin, y_pred_bin, zero_division=0)

# 3. Hiển thị báo cáo kết quả
print("\n" + "═"*60)
print("🏆 KẾT QUẢ ĐÁNH GIÁ: MULTI-VIEW MEAN POOLING (BASELINE)")
print("═"*60)
print(f"{'1. Regression Metrics (Điểm số)':<35}")
print(f"   - MAE (Sai số tuyệt đối): {mae:.4f}")
print(f"   - MSE (Sai số bình phương): {mse:.4f}")
print("-" * 60)
print(f"{'2. Classification Metrics (Phân loại)':<35}")
print(f"   - Accuracy (Độ chính xác):  {acc*100:.2f}%")
print(f"   - Precision (Độ chuẩn xác): {pre:.4f}")
print(f"   - Recall (Độ phủ):          {rec:.4f}")
print(f"   - F1-Score (Cân bằng):      {f1:.4f}")
print("═"*60)

# 4. Đối chiếu 10 mẫu đầu tiên để kiểm tra thực tế
print("\n🔍 Đối chiếu mẫu thực tế (10 mẫu đầu):")
print(f"{'ID':<4} | {'Thực tế':<10} | {'Điểm Gốc':<10} | {'Cosine Sim':<10} | {'Kết luận'}")
print("-" * 65)
for i in range(min(10, len(y_pred_test))):
    label_true = "Match" if y_true_bin[i] == 1 else "Mismatch"
    label_pred = "Match" if y_pred_bin[i] == 1 else "Mismatch"
    status = "✅" if y_true_bin[i] == y_pred_bin[i] else "❌"
    print(f"{i:<4} | {label_true:<10} | {y_true_reg[i]:<10.4f} | {y_pred_test[i]:<10.4f} | {status} {label_pred}")

# 5. Cập nhật kết quả vào file Excel tổng hợp
def upsert_baseline_excel(path, metrics):
    header = ["Encoder", "MAE", "MSE", "Accuracy", "Precision", "Recall", "F1-Score"]
    if not os.path.exists(path):
        wb = openpyxl.Workbook()
        ws = wb.active
        ws.title = "baselines"
        ws.append(header)
    else:
        wb = openpyxl.load_workbook(path)
        if "baselines" not in wb.sheetnames:
            ws = wb.create_sheet("baselines")
            ws.append(header)
        else:
            ws = wb["baselines"]
    
    # Thêm dòng kết quả mới
    ws.append([f"{MODEL_NAME}_mean_pooling"] + [round(float(m), 4) for m in metrics])
    wb.save(path)
    print(f"\n💾 Đã cập nhật kết quả vào: {path}")

EXCEL_PATH = "/kaggle/working/results_baselines_vn.xlsx"
upsert_baseline_excel(EXCEL_PATH, [mae, mse, acc, pre, rec, f1])

# Max Pooling

In [ ]:
!pip install -q sentence-transformers pandas tqdm openpyxl

import json
import os
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score

# ═══════════════════════════════════════════════════════════════════════════
# CẤU HÌNH ĐƯỜNG DẪN DATASET VIỆT NAM
# ═══════════════════════════════════════════════════════════════════════════
DATASET_ROOT = "/kaggle/input/datasets/thaiphonghuan/scoring/vietname-dataset-scoring - training"
CV_DIR = os.path.join(DATASET_ROOT, "cv")
JD_DIR = os.path.join(DATASET_ROOT, "jd")
TEST_MATCH_CSV = os.path.join(DATASET_ROOT, "dataset/test/match/test_match.csv")
TEST_MISMATCH_CSV = os.path.join(DATASET_ROOT, "dataset/test/mismatch/test_mismatch.csv")

CV_VIEWS = ['skills', 'education', 'year_experience', 'experience', 'responsibilities', 
            'projects', 'publications', 'achievement', 'certification', 'employment_history']
JD_VIEWS = ['skills', 'education', 'year_experience_requirement', 'experience_requirement', 'responsibilities']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Model đa ngôn ngữ hỗ trợ tiếng Việt cực tốt cho Bi-encoder
MODEL_NAME = 'BAAI/bge-base-en-v1.5'
sbert_model = SentenceTransformer(MODEL_NAME).to(device)

print(f"✅ Model loaded: {MODEL_NAME}")

In [ ]:
def process_view(view_data):
    """Xử lý missing_section thành special token [EMPTY_SECTION]; trả về (text, exists_flag)."""
    if view_data is None: return "[EMPTY_SECTION]", 0
    if isinstance(view_data, list):
        clean_list = [str(x) for x in view_data if str(x).lower() != "missing_section"]
        if not clean_list: return "[EMPTY_SECTION]", 0
        return ". ".join(clean_list), 1
    val = str(view_data).lower()
    if val in ("missing_section", "", "none"): return "[EMPTY_SECTION]", 0
    return str(view_data), 1

def run_max_pooling_on_csv():
    all_scores = []
    y_true_reg = []
    y_true_bin = []
    
    # Sử dụng các đường dẫn CSV từ Dataset Việt Nam đã cấu trúc
    configs = [(TEST_MATCH_CSV, 1), (TEST_MISMATCH_CSV, 0)]
    
    print(f"🚀 Đang chạy Multi-view Max Pooling (Bi-Encoder) cho dữ liệu Việt Nam...")
    
    for csv_path, b_label in configs:
        if not os.path.exists(csv_path):
            print(f"⚠️ Không tìm thấy: {csv_path}")
            continue
            
        df = pd.read_csv(csv_path)
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"CSV {b_label}"):
            # 1. Load JSON với logic Fallback (Quan trọng để tránh Sim = 0)
            with open(os.path.join(CV_DIR, row['cv']), 'r', encoding='utf-8-sig') as f:
                cv_obj = json.load(f)
                cv_j = cv_obj.get('resume_structured', cv_obj)
            with open(os.path.join(JD_DIR, row['jd']), 'r', encoding='utf-8-sig') as f:
                jd_obj = json.load(f)
                jd_j = jd_obj.get('job_description_structured', jd_obj)

            # 2. Trích xuất văn bản và mặt nạ cho từng View
            cv_texts, cv_masks = zip(*[process_view(cv_j.get(v)) for v in CV_VIEWS])
            jd_texts, jd_masks = zip(*[process_view(jd_j.get(v)) for v in JD_VIEWS])
            
            with torch.no_grad():
                # 3. Mã hóa độc lập từng View
                cv_emb = sbert_model.encode(list(cv_texts), convert_to_tensor=True) # [10, d]
                jd_emb = sbert_model.encode(list(jd_texts), convert_to_tensor=True) # [5, d]
                
                # 4. Thực hiện Masked Max Pooling
                # Tạo mask tensor [N, 1]
                cv_mask_t = torch.tensor(cv_masks, device=device).unsqueeze(1)
                jd_mask_t = torch.tensor(jd_masks, device=device).unsqueeze(1)
                
                # Với những view thiếu (mask=0), ta gán giá trị cực thấp (-1e9) 
                # để chúng không bao giờ được chọn làm giá trị Max
                z_cv = torch.max(cv_emb + (1 - cv_mask_t) * -1e9, dim=0)[0]
                z_jd = torch.max(jd_emb + (1 - jd_mask_t) * -1e9, dim=0)[0]
                
                # 5. Tính Cosine Similarity
                score = util.cos_sim(z_cv, z_jd).item()
            
            all_scores.append(score)
            y_true_reg.append(float(row['score']))
            y_true_bin.append(b_label)
            
    final_scores = np.array(all_scores)
    print(f"🔎 Sanity Check | Mean Sim: {final_scores.mean():.4f} | Max Sim: {final_scores.max():.4f}")
    
    return final_scores, np.array(y_true_reg), np.array(y_true_bin)

# Thực thi trích xuất kết quả
y_pred_test, y_true_reg, y_true_bin = run_max_pooling_on_csv()

In [ ]:
import openpyxl

# 1. Chuyển đổi điểm số sang nhãn nhị phân dựa trên ngưỡng 0.5
threshold = 0.5
y_pred_bin = (y_pred_test >= threshold).astype(int)

# 2. Tính toán trọn bộ Metrics
mae = mean_absolute_error(y_true_reg, y_pred_test)
mse = mean_squared_error(y_true_reg, y_pred_test)
acc = accuracy_score(y_true_bin, y_pred_bin)
pre = precision_score(y_true_bin, y_pred_bin, zero_division=0)
rec = recall_score(y_true_bin, y_pred_bin, zero_division=0)
f1  = f1_score(y_true_bin, y_pred_bin, zero_division=0)

# 3. Hiển thị báo cáo kết quả
print("\n" + "═"*60)
print("🏆 KẾT QUẢ ĐÁNH GIÁ: MULTI-VIEW MEAN POOLING (BASELINE)")
print("═"*60)
print(f"{'1. Regression Metrics (Điểm số)':<35}")
print(f"   - MAE (Sai số tuyệt đối): {mae:.4f}")
print(f"   - MSE (Sai số bình phương): {mse:.4f}")
print("-" * 60)
print(f"{'2. Classification Metrics (Phân loại)':<35}")
print(f"   - Accuracy (Độ chính xác):  {acc*100:.2f}%")
print(f"   - Precision (Độ chuẩn xác): {pre:.4f}")
print(f"   - Recall (Độ phủ):          {rec:.4f}")
print(f"   - F1-Score (Cân bằng):      {f1:.4f}")
print("═"*60)

# 4. Đối chiếu 10 mẫu đầu tiên để kiểm tra thực tế
print("\n🔍 Đối chiếu mẫu thực tế (10 mẫu đầu):")
print(f"{'ID':<4} | {'Thực tế':<10} | {'Điểm Gốc':<10} | {'Cosine Sim':<10} | {'Kết luận'}")
print("-" * 65)
for i in range(min(10, len(y_pred_test))):
    label_true = "Match" if y_true_bin[i] == 1 else "Mismatch"
    label_pred = "Match" if y_pred_bin[i] == 1 else "Mismatch"
    status = "✅" if y_true_bin[i] == y_pred_bin[i] else "❌"
    print(f"{i:<4} | {label_true:<10} | {y_true_reg[i]:<10.4f} | {y_pred_test[i]:<10.4f} | {status} {label_pred}")

# 5. Cập nhật kết quả vào file Excel tổng hợp
def upsert_baseline_excel(path, metrics):
    header = ["Encoder", "MAE", "MSE", "Accuracy", "Precision", "Recall", "F1-Score"]
    if not os.path.exists(path):
        wb = openpyxl.Workbook()
        ws = wb.active
        ws.title = "baselines"
        ws.append(header)
    else:
        wb = openpyxl.load_workbook(path)
        if "baselines" not in wb.sheetnames:
            ws = wb.create_sheet("baselines")
            ws.append(header)
        else:
            ws = wb["baselines"]
    
    # Thêm dòng kết quả mới
    ws.append([f"{MODEL_NAME}_mean_pooling"] + [round(float(m), 4) for m in metrics])
    wb.save(path)
    print(f"\n💾 Đã cập nhật kết quả vào: {path}")

EXCEL_PATH = "/kaggle/working/results_baselines_vn.xlsx"
upsert_baseline_excel(EXCEL_PATH, [mae, mse, acc, pre, rec, f1])

# Attention Pooling

In [ ]:
!pip install -q sentence-transformers pandas tqdm openpyxl

import json
import os
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score

# ═══════════════════════════════════════════════════════════════════════════
# CẤU HÌNH ĐƯỜNG DẪN DATASET VIỆT NAM (ĐẦY ĐỦ CHO TRAINING)
# ═══════════════════════════════════════════════════════════════════════════
DATASET_ROOT = "/kaggle/input/datasets/thaiphonghuan/scoring/vietname-dataset-scoring - training"
CV_DIR = os.path.join(DATASET_ROOT, "cv")
JD_DIR = os.path.join(DATASET_ROOT, "jd")
SPLIT_ROOT = os.path.join(DATASET_ROOT, "dataset")

# 1. Đường dẫn tập TRAIN
TRAIN_MATCH_CSV = os.path.join(SPLIT_ROOT, "train/match/train_match.csv")
TRAIN_MISMATCH_CSV = os.path.join(SPLIT_ROOT, "train/mismatch/train_mismatch.csv")

# 2. Đường dẫn tập VAL
VAL_MATCH_CSV = os.path.join(SPLIT_ROOT, "val/match/val_match.csv")
VAL_MISMATCH_CSV = os.path.join(SPLIT_ROOT, "val/mismatch/val_mismatch.csv")

# 3. Đường dẫn tập TEST
TEST_MATCH_CSV = os.path.join(SPLIT_ROOT, "test/match/test_match.csv")
TEST_MISMATCH_CSV = os.path.join(SPLIT_ROOT, "test/mismatch/test_mismatch.csv")

# Danh sách Views
CV_VIEWS = ['skills', 'education', 'year_experience', 'experience', 'responsibilities', 
            'projects', 'publications', 'achievement', 'certification', 'employment_history']
JD_VIEWS = ['skills', 'education', 'year_experience_requirement', 'experience_requirement', 'responsibilities']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Model nhúng
MODEL_NAME = 'BAAI/bge-base-en-v1.5'
sbert_model = SentenceTransformer(MODEL_NAME).to(device)

print(f"✅ Model loaded: {MODEL_NAME}")
print(f"📂 Dataset ready for Multi-task Training (Train/Val/Test split).")

In [ ]:
def process_view(view_data):
    if view_data is None: return "[EMPTY_SECTION]", 0
    if isinstance(view_data, list):
        clean_list = [str(x) for x in view_data if str(x).lower() != "missing_section"]
        if not clean_list: return "[EMPTY_SECTION]", 0
        return ". ".join(clean_list), 1
    val = str(view_data).lower()
    if val in ("missing_section", "", "none"): return "[EMPTY_SECTION]", 0
    return str(view_data), 1

def prepare_pooling_data_from_csv(match_csv, mismatch_csv, split_name):
    """Tiền tính toán Embeddings dựa trên các file CSV cụ thể."""
    cv_embs_list, cv_masks_list = [], []
    jd_embs_list, jd_masks_list = [], []
    reg_labels, bin_labels = [], []
    
    configs = [(match_csv, 1), (mismatch_csv, 0)]
    
    print(f"🚀 Đang xử lý tập dữ liệu: {split_name}")
    for csv_path, b_label in configs:
        if not os.path.exists(csv_path):
            print(f"⚠️ Cảnh báo: Không tìm thấy {csv_path}")
            continue
        
        df = pd.read_csv(csv_path)
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Split {split_name} Label {b_label}"):
            # Load JSON với fallback logic
            with open(os.path.join(CV_DIR, row['cv']), 'r', encoding='utf-8-sig') as f:
                cv_obj = json.load(f)
                cv_j = cv_obj.get('resume_structured', cv_obj)
            with open(os.path.join(JD_DIR, row['jd']), 'r', encoding='utf-8-sig') as f:
                jd_obj = json.load(f)
                jd_j = jd_obj.get('job_description_structured', jd_obj)

            cv_data = [process_view(cv_j.get(v)) for v in CV_VIEWS]
            jd_data = [process_view(jd_j.get(v)) for v in JD_VIEWS]
            
            cv_texts, cv_masks = zip(*cv_data)
            jd_texts, jd_masks = zip(*jd_data)
            
            with torch.no_grad():
                cv_emb = sbert_model.encode(list(cv_texts), convert_to_tensor=True).cpu().numpy()
                jd_emb = sbert_model.encode(list(jd_texts), convert_to_tensor=True).cpu().numpy()
            
            cv_embs_list.append(cv_emb)
            cv_masks_list.append(cv_masks)
            jd_embs_list.append(jd_emb)
            jd_masks_list.append(jd_masks)
            reg_labels.append(float(row['score']))
            bin_labels.append(b_label)
            
    return np.array(cv_embs_list), np.array(cv_masks_list), \
           np.array(jd_embs_list), np.array(jd_masks_list), \
           np.array(reg_labels), np.array(bin_labels)

# THỰC THI TRÍCH XUẤT
cv_h_train, cv_m_train, jd_h_train, jd_m_train, y_train, b_train = prepare_pooling_data_from_csv(TRAIN_MATCH_CSV, TRAIN_MISMATCH_CSV, "train")
cv_h_val,   cv_m_val,   jd_h_val,   jd_m_val,   y_val,   b_val   = prepare_pooling_data_from_csv(VAL_MATCH_CSV, VAL_MISMATCH_CSV, "val")
cv_h_test,  cv_m_test,  jd_h_test,  jd_m_test,  y_test,  b_test  = prepare_pooling_data_from_csv(TEST_MATCH_CSV, TEST_MISMATCH_CSV, "test")

print("\n✅ Hoàn tất tiền tính toán Embeddings cho Attention Pooling!")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader 

# 1. Định nghĩa Dataset để nạp các vector đã tính toán sẵn
class PrecomputedPoolingDataset(Dataset):
    def __init__(self, cv_h, cv_m, jd_h, jd_m, y_reg, y_bin):
        self.cv_h = torch.tensor(cv_h, dtype=torch.float32)
        self.cv_m = torch.tensor(cv_m, dtype=torch.float32)
        self.jd_h = torch.tensor(jd_h, dtype=torch.float32)
        self.jd_m = torch.tensor(jd_m, dtype=torch.float32)
        self.y_reg = torch.tensor(y_reg, dtype=torch.float32).view(-1, 1)
        self.y_bin = torch.tensor(y_bin, dtype=torch.float32).view(-1, 1)

    def __len__(self): return len(self.y_reg)

    def __getitem__(self, idx):
        return self.cv_h[idx], self.cv_m[idx], self.jd_h[idx], self.jd_m[idx], \
               self.y_reg[idx], self.y_bin[idx]

# 2. Định nghĩa kiến trúc Attention Pooling
class AttentionPoolingBaseline(nn.Module):
    def __init__(self, d_model=768): 
        super().__init__()
        # Lớp Linear để học trọng số cho từng view
        self.w = nn.Linear(d_model, 1, bias=False)

    def pool(self, h, mask):
        # h: [B, N_views, d], mask: [B, N_views]
        attn_scores = self.w(h).squeeze(-1) # [B, N_views]
        # Triệt tiêu các view trống bằng giá trị cực thấp trước khi tính Softmax
        attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        alpha = F.softmax(attn_scores, dim=-1) # Tính trọng số attention
        # Tổng có trọng số của các vector view: [B, 1, N_views] x [B, N_views, d] -> [B, d]
        z = torch.bmm(alpha.unsqueeze(1), h).squeeze(1) 
        return z

    def forward(self, cv_h, cv_m, jd_h, jd_m):
        z_cv = self.pool(cv_h, cv_m)
        z_jd = self.pool(jd_h, jd_m)
        # Tính Cosine Similarity giữa 2 vector đại diện
        cos_sim = F.cosine_similarity(z_cv, z_jd, dim=-1)
        return cos_sim.unsqueeze(-1)

# 3. Khởi tạo DataLoader
train_loader = DataLoader(PrecomputedPoolingDataset(cv_h_train, cv_m_train, jd_h_train, jd_m_train, y_train, b_train), batch_size=32, shuffle=True)
val_loader   = DataLoader(PrecomputedPoolingDataset(cv_h_val, cv_m_val, jd_h_val, jd_m_val, y_val, b_val), batch_size=32)
test_loader  = DataLoader(PrecomputedPoolingDataset(cv_h_test, cv_m_test, jd_h_test, jd_m_test, y_test, b_test), batch_size=32)

# 4. Khởi tạo Model, Loss và Optimizer
embedding_dim = cv_h_train.shape[-1] # Tự động lấy kích thước từ dữ liệu (768 hoặc 384)
model = AttentionPoolingBaseline(d_model=embedding_dim).to(device)

criterion_reg = nn.MSELoss()
criterion_bin = nn.BCELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

print(f"✅ Model Attention Pooling đã sẵn sàng! (D_model={embedding_dim})")

In [ ]:
epochs = 150
best_val_loss = float('inf')
ATTN_MODEL_PATH = "attn_pooling_best_vn.pth"

print("🔥 Bắt đầu huấn luyện Attention Pooling (Vietnamese)...")

for epoch in range(epochs):
    # --- GIAI ĐOẠN HUẤN LUYỆN ---
    model.train()
    total_train_loss = 0
    for cv_h, cv_m, jd_h, jd_m, yr, yb in train_loader:
        cv_h, cv_m, jd_h, jd_m = cv_h.to(device), cv_m.to(device), jd_h.to(device), jd_m.to(device)
        yr, yb = yr.to(device), yb.to(device)
        
        optimizer.zero_grad()
        # Chạy model ra similarity
        pred = model(cv_h, cv_m, jd_h, jd_m)
        # Ép về [0, 1] để tính BCE Loss an toàn
        pred_c = torch.clamp(pred, 0, 1)
        
        # ✅ CẬP NHẬT: Thay 0.2 thành 0.5
        loss = criterion_reg(pred_c, yr) + 0.5 * criterion_bin(pred_c, yb)
        
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
    
    # --- GIAI ĐOẠN ĐÁNH GIÁ ---
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for cv_h, cv_m, jd_h, jd_m, yr, yb in val_loader:
            cv_h, cv_m, jd_h, jd_m = cv_h.to(device), cv_m.to(device), jd_h.to(device), jd_m.to(device)
            yr, yb = yr.to(device), yb.to(device)
            out = model(cv_h, cv_m, jd_h, jd_m)
            out_c = torch.clamp(out, 0, 1)
            
            # ✅ CẬP NHẬT: Thay 0.2 thành 0.5 (Phải đồng bộ với lúc Train)
            val_loss += (criterion_reg(out_c, yr) + 0.5 * criterion_bin(out_c, yb)).item()
    
    avg_val = val_loss / len(val_loader)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d} | Train Loss: {total_train_loss/len(train_loader):.6f} | Val Loss: {avg_val:.6f}")
    
    # Lưu checkpoint dựa trên Val Loss tốt nhất
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), ATTN_MODEL_PATH)

print(f"✅ Hoàn tất huấn luyện! Best Val Loss: {best_val_loss:.6f}")

In [ ]:
import torch
import numpy as np
import pandas as pd
import openpyxl
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score

# ═══════════════════════════════════════════════════════════════════════════
# 1. NẠP MODEL TỐT NHẤT
# ═══════════════════════════════════════════════════════════════════════════
# Đảm bảo tên file khớp với Cell Training (attn_pooling_best_vn.pth)
MODEL_WEIGHTS = "attn_pooling_best_vn.pth"

if os.path.exists(MODEL_WEIGHTS):
    model.load_state_dict(torch.load(MODEL_WEIGHTS, map_location=device))
    model.eval()
    print(f"📂 Loaded best weights from {MODEL_WEIGHTS}")
else:
    print(f"⚠️ Cảnh báo: Không tìm thấy file {MODEL_WEIGHTS}, đang dùng trọng số hiện tại.")

y_preds_test = []
y_true_reg_test = []
y_true_bin_test = []

# ═══════════════════════════════════════════════════════════════════════════
# 2. THỰC THI DỰ ĐOÁN
# ═══════════════════════════════════════════════════════════════════════════
print("🚀 Đang thực hiện đánh giá trên tập Test (Attention Pooling)...")

with torch.no_grad():
    for cv_h, cv_m, jd_h, jd_m, yr, yb in tqdm(test_loader, desc="Testing"):
        cv_h, cv_m = cv_h.to(device), cv_m.to(device)
        jd_h, jd_m = jd_h.to(device), jd_m.to(device)
        
        # Forward pass
        outputs = model(cv_h, cv_m, jd_h, jd_m)
        
        # Ép điểm về khoảng [0, 1] 
        outputs_clamped = torch.clamp(outputs, 0, 1)
        
        y_preds_test.extend(outputs_clamped.cpu().numpy().flatten())
        y_true_reg_test.extend(yr.numpy().flatten())
        y_true_bin_test.extend(yb.numpy().flatten())

# Chuyển sang mảng numpy
y_preds_test = np.array(y_preds_test)
y_true_reg_test = np.array(y_true_reg_test)
y_true_bin_test = np.array(y_true_bin_test)

# Chuyển sang nhãn nhị phân
threshold = 0.5
y_preds_bin_test = (y_preds_test >= threshold).astype(int)

# ═══════════════════════════════════════════════════════════════════════════
# 3. TÍNH TOÁN VÀ HIỂN THỊ METRICS
# ═══════════════════════════════════════════════════════════════════════════
mae = mean_absolute_error(y_true_reg_test, y_preds_test)
mse = mean_squared_error(y_true_reg_test, y_preds_test)
acc = accuracy_score(y_true_bin_test, y_preds_bin_test)
pre = precision_score(y_true_bin_test, y_preds_bin_test, zero_division=0)
rec = recall_score(y_true_bin_test, y_preds_bin_test, zero_division=0)
f1  = f1_score(y_true_bin_test, y_preds_bin_test, zero_division=0)

print("\n" + "═"*60)
print("🏆 KẾT QUẢ ĐÁNH GIÁ: ATTENTION POOLING BASELINE")
print("═"*60)
print(f"{'1. Regression Metrics:':<30}")
print(f"   - MAE: {mae:.4f}")
print(f"   - MSE: {mse:.4f}")
print("-" * 60)
print(f"{'2. Classification Metrics:':<30}")
print(f"   - Accuracy:  {acc*100:.2f}%")
print(f"   - Precision: {pre:.4f}")
print(f"   - Recall:    {rec:.4f}")
print(f"   - F1-Score:  {f1:.4f}")
print("═"*60)

# ═══════════════════════════════════════════════════════════════════════════
# 4. LƯU VÀO EXCEL (DÙNG CHUNG FILE VỚI CÁC BASELINE KHÁC)
# ═══════════════════════════════════════════════════════════════════════════
def upsert_baseline_excel(path, model_desc, metrics):
    header = ["Encoder", "MAE", "MSE", "Accuracy", "Precision", "Recall", "F1-Score"]
    if not os.path.exists(path):
        wb = openpyxl.Workbook(); ws = wb.active; ws.title = "baselines"; ws.append(header)
    else:
        wb = openpyxl.load_workbook(path)
        ws = wb["baselines"] if "baselines" in wb.sheetnames else wb.create_sheet("baselines")
    
    ws.append([model_desc] + [round(float(m), 4) for m in metrics])
    wb.save(path)
    print(f"💾 Kết quả đã được lưu vào: {path}")

EXCEL_PATH = "/kaggle/working/results_baselines_vn.xlsx"
upsert_baseline_excel(EXCEL_PATH, f"{MODEL_NAME}_attention_pooling", [mae, mse, acc, pre, rec, f1])